In [1]:
import random
import re
from collections import Counter

import torch
import torch.nn as nn
import torch.optim as optim

from datasets import load_dataset


# Make results more reproducible
random.seed(42)
torch.manual_seed(42)


# Use GPU if Colab provides one.
# Otherwise use CPU.
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)

Using device: cpu


In [2]:
# Load the OPUS Books English-French translation dataset
dataset = load_dataset(
    "Helsinki-NLP/opus_books",
    "en-fr"
)

print(dataset)

print(
    "Total sentence pairs:",
    len(dataset["train"])
)

DatasetDict({
    train: Dataset({
        features: ['id', 'translation'],
        num_rows: 127085
    })
})
Total sentence pairs: 127085


In [3]:
def normalize_text(text):

    # Convert uppercase letters to lowercase
    text = text.lower()

    # Normalize curly apostrophes
    text = text.replace("’", "'")
    text = text.replace("‘", "'")

    # Normalize different dash characters
    text = text.replace("–", "-")
    text = text.replace("—", "-")

    # Remove unnecessary spaces
    text = text.strip()

    return text

In [4]:
def tokenize(text):

    # Clean the text first
    text = normalize_text(text)

    # Separate words and punctuation
    tokens = re.findall(
        r"\w+|[^\w\s]",
        text,
        flags=re.UNICODE
    )

    return tokens

In [5]:
tokenized_pairs = []


for example in dataset["train"]:

    english = example["translation"]["en"]
    french = example["translation"]["fr"]

    # Convert both sentences into tokens
    english_tokens = tokenize(english)
    french_tokens = tokenize(french)

    # Keep only relatively short sentences
    if (
        0 < len(english_tokens) <= 12
        and
        0 < len(french_tokens) <= 15
    ):

        tokenized_pairs.append(
            (
                english_tokens,
                french_tokens
            )
        )


print(
    "Short sentence pairs:",
    len(tokenized_pairs)
)

Short sentence pairs: 30862


In [6]:
# Shuffle before selecting examples
random.shuffle(tokenized_pairs)


MAX_PAIRS = 8000


# Use at most 8000 pairs
pairs = tokenized_pairs[:MAX_PAIRS]


print(
    "Sentence pairs used:",
    len(pairs)
)

Sentence pairs used: 8000


In [7]:
# 80% training
train_size = int(
    0.80 * len(pairs)
)

# 10% validation
validation_size = int(
    0.10 * len(pairs)
)


train_pairs = pairs[:train_size]

validation_pairs = pairs[
    train_size:
    train_size + validation_size
]

test_pairs = pairs[
    train_size + validation_size:
]


print(
    "Training examples:",
    len(train_pairs)
)

print(
    "Validation examples:",
    len(validation_pairs)
)

print(
    "Test examples:",
    len(test_pairs)
)

Training examples: 6400
Validation examples: 800
Test examples: 800


In [8]:
english_word_counts = Counter()
french_word_counts = Counter()


# Count words appearing in training examples
for english_tokens, french_tokens in train_pairs:

    english_word_counts.update(
        english_tokens
    )

    french_word_counts.update(
        french_tokens
    )


print(
    "Unique English tokens:",
    len(english_word_counts)
)

print(
    "Unique French tokens:",
    len(french_word_counts)
)

Unique English tokens: 6070
Unique French tokens: 7138


In [9]:
minimum_frequency = 2


# Special tokens used by our model
special_tokens = [
    "<PAD>",
    "<UNK>",
    "<START>",
    "<END>"
]


# ------------------------------
# English vocabulary
# ------------------------------

english_vocabulary = (
    special_tokens.copy()
)


for word, count in english_word_counts.items():

    if count >= minimum_frequency:

        english_vocabulary.append(
            word
        )


# ------------------------------
# French vocabulary
# ------------------------------

french_vocabulary = (
    special_tokens.copy()
)


for word, count in french_word_counts.items():

    if count >= minimum_frequency:

        french_vocabulary.append(
            word
        )


print(
    "English vocabulary size:",
    len(english_vocabulary)
)

print(
    "French vocabulary size:",
    len(french_vocabulary)
)

English vocabulary size: 2642
French vocabulary size: 2780


In [10]:
# English word -> ID
english_word_to_id = {
    word: index
    for index, word
    in enumerate(english_vocabulary)
}


# English ID -> word
english_id_to_word = {
    index: word
    for word, index
    in english_word_to_id.items()
}


# French word -> ID
french_word_to_id = {
    word: index
    for index, word
    in enumerate(french_vocabulary)
}


# French ID -> word
french_id_to_word = {
    index: word
    for word, index
    in french_word_to_id.items()
}


print(
    "English <UNK>:",
    english_word_to_id["<UNK>"]
)

print(
    "French <START>:",
    french_word_to_id["<START>"]
)

print(
    "French <END>:",
    french_word_to_id["<END>"]
)

English <UNK>: 1
French <START>: 2
French <END>: 3


In [11]:
def prepare_training_pair(
    english_tokens,
    french_tokens
):

    # =====================================
    # Encoder input
    # =====================================

    # English sentence ends with <END>
    encoder_tokens = (
        english_tokens
        +
        ["<END>"]
    )


    encoder_ids = [
        english_word_to_id.get(
            word,
            english_word_to_id["<UNK>"]
        )
        for word in encoder_tokens
    ]


    # =====================================
    # Decoder input
    # =====================================

    # Decoder begins with <START>
    decoder_input_tokens = (
        ["<START>"]
        +
        french_tokens
    )


    decoder_input_ids = [
        french_word_to_id.get(
            word,
            french_word_to_id["<UNK>"]
        )
        for word in decoder_input_tokens
    ]


    # =====================================
    # Decoder target
    # =====================================

    # Decoder should eventually predict <END>
    decoder_target_tokens = (
        french_tokens
        +
        ["<END>"]
    )


    decoder_target_ids = [
        french_word_to_id.get(
            word,
            french_word_to_id["<UNK>"]
        )
        for word in decoder_target_tokens
    ]


    # Convert Python lists to PyTorch tensors

    encoder_tensor = torch.tensor(
        encoder_ids,
        dtype=torch.long,
        device=device
    )


    decoder_input_tensor = torch.tensor(
        decoder_input_ids,
        dtype=torch.long,
        device=device
    )


    decoder_target_tensor = torch.tensor(
        decoder_target_ids,
        dtype=torch.long,
        device=device
    )


    return (
        encoder_tensor,
        decoder_input_tensor,
        decoder_target_tensor
    )

In [12]:
class Encoder(nn.Module):

    def __init__(
        self,
        vocab_size,
        embedding_size,
        hidden_size
    ):

        super().__init__()


        # Convert English word IDs
        # into learned vectors
        self.embedding = nn.Embedding(
            vocab_size,
            embedding_size
        )


        # Process English words sequentially
        self.lstm = nn.LSTM(
            input_size=embedding_size,
            hidden_size=hidden_size
        )


    def forward(
        self,
        encoder_input
    ):

        # Word IDs -> embeddings
        embedded = self.embedding(
            encoder_input
        )


        # encoder_outputs contains:
        # h1, h2, h3 ... hN
        #
        # hidden_state = final hN
        # cell_state   = final cN

        encoder_outputs, (
            hidden_state,
            cell_state
        ) = self.lstm(
            embedded
        )


        return (
            encoder_outputs,
            hidden_state,
            cell_state
        )

In [13]:
class Attention(nn.Module):

    def __init__(
        self,
        hidden_size
    ):

        super().__init__()


        # Transform encoder hidden states
        self.encoder_layer = nn.Linear(
            hidden_size,
            hidden_size
        )


        # Transform current decoder state
        self.decoder_layer = nn.Linear(
            hidden_size,
            hidden_size
        )


        # Produce one relevance score
        # for each encoder state
        self.score_layer = nn.Linear(
            hidden_size,
            1
        )


    def forward(
        self,
        decoder_hidden,
        encoder_outputs
    ):

        # decoder_hidden:
        # [1, hidden_size]
        #
        # encoder_outputs:
        # [English_length, hidden_size]


        # Remove layer dimension
        decoder_hidden = (
            decoder_hidden.squeeze(0)
        )


        # Transform all English encoder states
        encoder_part = (
            self.encoder_layer(
                encoder_outputs
            )
        )


        # Transform current decoder state
        decoder_part = (
            self.decoder_layer(
                decoder_hidden
            )
        )


        # Combine encoder + decoder information
        combined = torch.tanh(
            encoder_part
            +
            decoder_part
        )


        # Calculate raw attention scores
        scores = self.score_layer(
            combined
        ).squeeze(1)


        # Convert scores to weights
        # that add up to 1
        attention_weights = torch.softmax(
            scores,
            dim=0
        )


        # Weight each encoder hidden state
        weighted_outputs = (
            attention_weights.unsqueeze(1)
            *
            encoder_outputs
        )


        # Combine weighted states
        # into one context vector
        context_vector = torch.sum(
            weighted_outputs,
            dim=0,
            keepdim=True
        )


        return (
            context_vector,
            attention_weights
        )

In [14]:
class AttentionDecoder(nn.Module):

    def __init__(
        self,
        vocab_size,
        embedding_size,
        hidden_size
    ):

        super().__init__()


        # French word embeddings
        self.embedding = nn.Embedding(
            vocab_size,
            embedding_size
        )


        # Attention layer
        self.attention = Attention(
            hidden_size
        )


        # LSTM receives:
        #
        # French embedding
        # +
        # attention context vector
        self.lstm = nn.LSTM(
            input_size=(
                embedding_size
                +
                hidden_size
            ),
            hidden_size=hidden_size
        )


        # Convert hidden state into
        # French vocabulary scores
        self.output_layer = nn.Linear(
            hidden_size,
            vocab_size
        )


    def forward(
        self,
        current_token,
        hidden_state,
        cell_state,
        encoder_outputs
    ):

        # ---------------------------------
        # 1. French token embedding
        # ---------------------------------

        embedded = self.embedding(
            current_token
        )


        # ---------------------------------
        # 2. Calculate attention
        # ---------------------------------

        (
            context_vector,
            attention_weights
        ) = self.attention(
            hidden_state,
            encoder_outputs
        )


        # ---------------------------------
        # 3. Combine:
        #
        # French embedding
        # +
        # English context
        # ---------------------------------

        decoder_input = torch.cat(
            (
                embedded,
                context_vector
            ),
            dim=1
        )


        # ---------------------------------
        # 4. Decoder LSTM
        # ---------------------------------

        decoder_output, (
            hidden_state,
            cell_state
        ) = self.lstm(
            decoder_input,
            (
                hidden_state,
                cell_state
            )
        )


        # ---------------------------------
        # 5. French word scores
        # ---------------------------------

        logits = self.output_layer(
            decoder_output
        )


        return (
            logits,
            hidden_state,
            cell_state,
            attention_weights
        )

In [15]:
# Size of each learned word embedding
embedding_size = 64


# Number of values in LSTM hidden/cell state
hidden_size = 128


encoder = Encoder(
    vocab_size=len(
        english_vocabulary
    ),
    embedding_size=embedding_size,
    hidden_size=hidden_size
).to(device)


decoder = AttentionDecoder(
    vocab_size=len(
        french_vocabulary
    ),
    embedding_size=embedding_size,
    hidden_size=hidden_size
).to(device)


print(encoder)

print()

print(decoder)

Encoder(
  (embedding): Embedding(2642, 64)
  (lstm): LSTM(64, 128)
)

AttentionDecoder(
  (embedding): Embedding(2780, 64)
  (attention): Attention(
    (encoder_layer): Linear(in_features=128, out_features=128, bias=True)
    (decoder_layer): Linear(in_features=128, out_features=128, bias=True)
    (score_layer): Linear(in_features=128, out_features=1, bias=True)
  )
  (lstm): LSTM(192, 128)
  (output_layer): Linear(in_features=128, out_features=2780, bias=True)
)


In [16]:
# Cross entropy is used because
# each decoder step chooses one word
# from the French vocabulary.
loss_function = nn.CrossEntropyLoss()


# One optimizer trains BOTH
# encoder and decoder.
all_parameters = (
    list(encoder.parameters())
    +
    list(decoder.parameters())
)


optimizer = optim.Adam(
    all_parameters,
    lr=0.001
)

In [17]:
def train_one_pair(
    english_tokens,
    french_tokens
):

    # Clear gradients from previous example
    optimizer.zero_grad()


    # Convert words to tensors
    (
        encoder_tensor,
        decoder_input_tensor,
        decoder_target_tensor
    ) = prepare_training_pair(
        english_tokens,
        french_tokens
    )


    # =====================================
    # ENCODER
    # =====================================

    (
        encoder_outputs,
        encoder_hidden,
        encoder_cell
    ) = encoder(
        encoder_tensor
    )


    # Decoder starts with
    # encoder final memory
    decoder_hidden = encoder_hidden
    decoder_cell = encoder_cell


    # Store decoder predictions
    all_logits = []


    # =====================================
    # DECODER
    # =====================================

    # Teacher forcing:
    # give correct previous French token
    # at every decoder step.
    for step in range(
        len(decoder_input_tensor)
    ):

        current_token = (
            decoder_input_tensor[
                step
            ].unsqueeze(0)
        )


        (
            logits,
            decoder_hidden,
            decoder_cell,
            attention_weights
        ) = decoder(
            current_token,
            decoder_hidden,
            decoder_cell,
            encoder_outputs
        )


        # logits shape:
        # [1, French_vocab_size]
        #
        # Remove first dimension
        all_logits.append(
            logits.squeeze(0)
        )


    # Combine all decoder predictions
    all_logits = torch.stack(
        all_logits
    )


    # Compare predictions with
    # correct French target tokens
    loss = loss_function(
        all_logits,
        decoder_target_tensor
    )


    # Calculate gradients through:
    #
    # decoder
    # attention
    # encoder
    loss.backward()


    # Prevent extremely large
    # LSTM gradients
    torch.nn.utils.clip_grad_norm_(
        all_parameters,
        max_norm=1.0
    )


    # Actually update model weights
    optimizer.step()


    return loss.item()

In [18]:
def calculate_validation_loss(
    validation_pairs
):

    encoder.eval()
    decoder.eval()

    total_loss = 0


    # No gradients needed
    with torch.no_grad():

        for (
            english_tokens,
            french_tokens
        ) in validation_pairs:

            (
                encoder_tensor,
                decoder_input_tensor,
                decoder_target_tensor
            ) = prepare_training_pair(
                english_tokens,
                french_tokens
            )


            # Encoder
            (
                encoder_outputs,
                decoder_hidden,
                decoder_cell
            ) = encoder(
                encoder_tensor
            )


            all_logits = []


            # Decoder using teacher forcing
            for step in range(
                len(decoder_input_tensor)
            ):

                current_token = (
                    decoder_input_tensor[
                        step
                    ].unsqueeze(0)
                )


                (
                    logits,
                    decoder_hidden,
                    decoder_cell,
                    _
                ) = decoder(
                    current_token,
                    decoder_hidden,
                    decoder_cell,
                    encoder_outputs
                )


                all_logits.append(
                    logits.squeeze(0)
                )


            all_logits = torch.stack(
                all_logits
            )


            loss = loss_function(
                all_logits,
                decoder_target_tensor
            )


            total_loss += (
                loss.item()
            )


    return (
        total_loss
        /
        len(validation_pairs)
    )

In [19]:
number_of_epochs = 10


for epoch in range(
    number_of_epochs
):

    # Put models into training mode
    encoder.train()
    decoder.train()


    # Shuffle sentence order
    random.shuffle(
        train_pairs
    )


    total_training_loss = 0


    # Train on every sentence pair
    for index, (
        english_tokens,
        french_tokens
    ) in enumerate(train_pairs):


        loss = train_one_pair(
            english_tokens,
            french_tokens
        )


        total_training_loss += loss


        # Show occasional progress
        if (
            index + 1
        ) % 1000 == 0:

            print(
                f"Epoch {epoch + 1} "
                f"| Example {index + 1}"
                f"/{len(train_pairs)}"
            )


    average_training_loss = (
        total_training_loss
        /
        len(train_pairs)
    )


    # Measure validation loss
    validation_loss = (
        calculate_validation_loss(
            validation_pairs
        )
    )


    print(
        f"\nEpoch {epoch + 1:2d}"
        f" | Training Loss: "
        f"{average_training_loss:.4f}"
        f" | Validation Loss: "
        f"{validation_loss:.4f}\n"
    )

Epoch 1 | Example 1000/6400
Epoch 1 | Example 2000/6400
Epoch 1 | Example 3000/6400
Epoch 1 | Example 4000/6400
Epoch 1 | Example 5000/6400
Epoch 1 | Example 6000/6400

Epoch  1 | Training Loss: 4.0882 | Validation Loss: 3.2788

Epoch 2 | Example 1000/6400
Epoch 2 | Example 2000/6400
Epoch 2 | Example 3000/6400
Epoch 2 | Example 4000/6400
Epoch 2 | Example 5000/6400
Epoch 2 | Example 6000/6400

Epoch  2 | Training Loss: 3.2150 | Validation Loss: 2.9557

Epoch 3 | Example 1000/6400
Epoch 3 | Example 2000/6400
Epoch 3 | Example 3000/6400
Epoch 3 | Example 4000/6400
Epoch 3 | Example 5000/6400
Epoch 3 | Example 6000/6400

Epoch  3 | Training Loss: 2.7548 | Validation Loss: 2.8804

Epoch 4 | Example 1000/6400
Epoch 4 | Example 2000/6400
Epoch 4 | Example 3000/6400
Epoch 4 | Example 4000/6400
Epoch 4 | Example 5000/6400
Epoch 4 | Example 6000/6400

Epoch  4 | Training Loss: 2.3872 | Validation Loss: 2.8533

Epoch 5 | Example 1000/6400
Epoch 5 | Example 2000/6400
Epoch 5 | Example 3000/6400


In [20]:
def translate(
    english_sentence,
    max_output_length=20
):

    # Prepare model for inference
    encoder.eval()
    decoder.eval()


    # Tokenize user's English sentence
    english_tokens = tokenize(
        english_sentence
    )


    # Add <END>
    encoder_tokens = (
        english_tokens
        +
        ["<END>"]
    )


    # Convert English words to IDs
    encoder_ids = [
        english_word_to_id.get(
            word,
            english_word_to_id["<UNK>"]
        )
        for word in encoder_tokens
    ]


    encoder_tensor = torch.tensor(
        encoder_ids,
        dtype=torch.long,
        device=device
    )


    predicted_words = []

    saved_attention = []


    with torch.no_grad():

        # ------------------------------
        # Encoder
        # ------------------------------

        (
            encoder_outputs,
            decoder_hidden,
            decoder_cell
        ) = encoder(
            encoder_tensor
        )


        # ------------------------------
        # Decoder starts with <START>
        # ------------------------------

        current_token = torch.tensor(
            [
                french_word_to_id[
                    "<START>"
                ]
            ],
            dtype=torch.long,
            device=device
        )


        # Generate French words
        # one at a time
        for _ in range(
            max_output_length
        ):

            (
                logits,
                decoder_hidden,
                decoder_cell,
                attention_weights
            ) = decoder(
                current_token,
                decoder_hidden,
                decoder_cell,
                encoder_outputs
            )


            # Pick highest-scoring word
            predicted_id = torch.argmax(
                logits[0]
            ).item()


            predicted_word = (
                french_id_to_word[
                    predicted_id
                ]
            )


            # Save attention so we can
            # visualize it later
            saved_attention.append(
                attention_weights.cpu()
            )


            # Stop if model predicts <END>
            if predicted_word == "<END>":
                break


            predicted_words.append(
                predicted_word
            )


            # Feed model's own prediction
            # into next decoder step
            current_token = torch.tensor(
                [predicted_id],
                dtype=torch.long,
                device=device
            )


    return (
        predicted_words,
        english_tokens,
        saved_attention
    )

In [21]:
# Pick 10 random unseen test examples
examples = random.sample(
    test_pairs,
    10
)


for (
    english_tokens,
    correct_french_tokens
) in examples:

    english_sentence = " ".join(
        english_tokens
    )


    (
        predicted_words,
        _,
        _
    ) = translate(
        english_sentence
    )


    print(
        "English  :",
        english_sentence
    )

    print(
        "Expected :",
        " ".join(
            correct_french_tokens
        )
    )

    print(
        "Predicted:",
        " ".join(
            predicted_words
        )
    )

    print(
        "-" * 70
    )

English  : " at close quarters ? " i repeated .
Expected : « corps à corps ? répétai - je .
Predicted: - - a fait , je <UNK> ? demandai - je .
----------------------------------------------------------------------
English  : then the memory of the bertaux came back to her .
Expected : alors le souvenir des bertaux lui arriva .
Predicted: puis le niveau de lampe sous la <UNK> .
----------------------------------------------------------------------
English  : " my jailers and my hangmen !
Expected : - - mes geôliers ! mes bourreaux !
Predicted: ma chere <UNK> et mon <UNK> !
----------------------------------------------------------------------
English  : " when ? "
Expected : - - quand ?
Predicted: - - quand ?
----------------------------------------------------------------------
English  : there was silence .
Expected : il y eut un silence .
Predicted: le silence recommença .
----------------------------------------------------------------------
English  : rassi veut donc me déshonorer 

In [22]:
torch.save(
    {
        # Model weights
        "encoder_state_dict":
            encoder.state_dict(),

        "decoder_state_dict":
            decoder.state_dict(),


        # Vocabulary mappings
        "english_word_to_id":
            english_word_to_id,

        "english_id_to_word":
            english_id_to_word,

        "french_word_to_id":
            french_word_to_id,

        "french_id_to_word":
            french_id_to_word,


        # Model configuration
        "embedding_size":
            embedding_size,

        "hidden_size":
            hidden_size,

        "english_vocab_size":
            len(
                english_vocabulary
            ),

        "french_vocab_size":
            len(
                french_vocabulary
            )
    },

    "translation_attention_model.pt"
)


print(
    "Model saved as:"
    " translation_attention_model.pt"
)

Model saved as: translation_attention_model.pt
